# Fixture ID Mapping (Sportradar ↔ OddsJam)

Build a 1:1 crosswalk between Sportradar `sport_event_id` and OddsJam `fixture_id`.

**Matching rules**
- Collapse consensus to match-level player pairs (moneyline first; fill from other player markets if needed)
- Normalize names to sorted underscore tokens on both sides
- Require both players to match as a set
- Keep candidates within ±12 hours (`first_event_time` vs `start_date`)
- Accept only mutual rank-1 matches with a clear time margin

Matching helpers live in `refined_tables/fixture_id_mapping/build.py` for later backfill extraction.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "injestion").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError(f"Could not find project root from {cwd}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from injestion.core.bq import get_client
from injestion.core.env import load_env

load_env()
bq_client = get_client()
print(f"Project root: {project_root}")
print(f"BQ project: {bq_client.project}")

In [ ]:
from refined_tables import get_table_id

FIXTURE_STATS_TABLE_ID = get_table_id("fixture_stats")
CONSENSUS_TABLE_ID = get_table_id("consensus")
MAPPING_TABLE_ID = get_table_id("fixture_id_mapping")

print(f"fixture_stats: {FIXTURE_STATS_TABLE_ID}")
print(f"consensus:     {CONSENSUS_TABLE_ID}")
print(f"mapping:       {MAPPING_TABLE_ID}")

## Load source tables

In [ ]:
fixture_stats_df = bq_client.query(
    f"""
    SELECT
      sport_event_id,
      home_competitor_name,
      away_competitor_name,
      first_event_time
    FROM `{FIXTURE_STATS_TABLE_ID}`
    """
).to_dataframe()

print(f"fixture_stats rows: {len(fixture_stats_df):,}")
fixture_stats_df.head()

In [ ]:
consensus_df = bq_client.query(
    f"""
    SELECT
      fixture_id,
      start_date,
      market,
      name,
      normalized_selection,
      normalized_selection_key
    FROM `{CONSENSUS_TABLE_ID}`
    """
).to_dataframe()

print(f"consensus rows: {len(consensus_df):,}")
consensus_df.head()

## Build match-level frames + join

In [ ]:
from refined_tables.fixture_id_mapping import (
    assert_one_to_one,
    build_oddsjam_fixture_pairs,
    format_accepted_for_upload,
    match_fixtures,
    normalize_player_name,
    prepare_sportradar_matches,
)

# Sanity-check name normalization
for raw in ["Wong, Hong Yi Cody", "Abdullah Shelbayh", "abdullah_shelbayh"]:
    print(f"{raw!r:40s} -> {normalize_player_name(raw)}")

In [ ]:
sr_matches = prepare_sportradar_matches(fixture_stats_df)
oj_pairs, oj_insufficient = build_oddsjam_fixture_pairs(consensus_df)

print(f"Sportradar match rows:     {len(sr_matches):,}")
print(f"OddsJam pair rows:         {len(oj_pairs):,}")
print(f"OddsJam insufficient:      {len(oj_insufficient):,}")
if not oj_insufficient.empty:
    display(oj_insufficient.head(10))

display(sr_matches.head())
display(oj_pairs.head())

In [ ]:
TIME_WINDOW_HOURS = 12.0
TIME_MARGIN_MINUTES = 120.0

results = match_fixtures(
    sr_matches,
    oj_pairs,
    time_window_hours=TIME_WINDOW_HOURS,
    time_margin_minutes=TIME_MARGIN_MINUTES,
)

accepted_raw = results["accepted"]
ambiguous = results["ambiguous"]
unmatched_sr = results["unmatched_sr"]
candidates = results["candidates"]

print(f"Candidates in window: {len(candidates):,}")
print(f"Accepted:             {len(accepted_raw):,}")
print(f"Ambiguous:            {len(ambiguous):,}")
print(f"Unmatched SR:         {len(unmatched_sr):,}")

accepted_df = format_accepted_for_upload(accepted_raw)
assert_one_to_one(accepted_df)
print("1:1 uniqueness check passed")
accepted_df.head(20)

## QC

In [ ]:
n_sr = len(sr_matches)
n_oj = len(oj_pairs)
n_acc = len(accepted_df)

print(f"SR coverage: {n_acc / n_sr:.1%} ({n_acc:,} / {n_sr:,})" if n_sr else "SR coverage: n/a")
print(f"OJ coverage: {n_acc / n_oj:.1%} ({n_acc:,} / {n_oj:,})" if n_oj else "OJ coverage: n/a")
print(f"Insufficient OJ fixtures: {len(oj_insufficient):,}")

if not accepted_df.empty:
    deltas = accepted_df["time_delta_minutes"].abs()
    print("\n|time_delta_minutes| summary (accepted):")
    print(deltas.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

if not ambiguous.empty:
    print(f"\nAmbiguous sample ({len(ambiguous):,} rows):")
    display(
        ambiguous[
            [
                c
                for c in [
                    "sport_event_id",
                    "fixture_id",
                    "home_competitor_name",
                    "away_competitor_name",
                    "abs_time_delta_minutes",
                    "rank_by_sr",
                    "rank_by_oj",
                ]
                if c in ambiguous.columns
            ]
        ].head(20)
    )

In [ ]:
# Spot-check random accepted links
if not accepted_df.empty:
    sample_n = min(20, len(accepted_df))
    display(
        accepted_df.sample(sample_n, random_state=42)[
            [
                "sport_event_id",
                "fixture_id",
                "home_competitor_name",
                "away_competitor_name",
                "oj_player_a",
                "oj_player_b",
                "first_event_time",
                "start_date",
                "time_delta_minutes",
            ]
        ].sort_values("time_delta_minutes")
    )
else:
    print("No accepted rows to spot-check.")

## Upload accepted crosswalk

Requires `BIGQUERY_REFINED_FIXTURE_ID_MAPPING_TABLE_ID` in `.env`.
Uses `WRITE_TRUNCATE` (full rebuild).

In [ ]:
from refined_tables import replace_table
from refined_tables.schema import fixture_id_mapping as schema_mapping

assert_one_to_one(accepted_df)

if accepted_df.empty:
    raise ValueError("Refusing to upload an empty accepted crosswalk.")

job = replace_table(
    bq_client,
    MAPPING_TABLE_ID,
    accepted_df,
    schema=schema_mapping.get_schema(),
)
print(f"Uploaded {len(accepted_df):,} rows to {MAPPING_TABLE_ID}")
job